# 02 - Popularity Prediction
**Spotify Data Mining | CISC 4631 | Group 3**

### Research Questions
1. can audio features predict if a song will be **globally popular**?
2. can audio features predict if a song will be popular **within its genre**?
3. do same features drive both, or does genre context change what matters?
4. does the task get easier if we collapse popularity to **binary**?

### Outline
1. Load shared data
2. Preprocess, split (60/20/20), feature selection, class-balance check
3. Baselines: KNN, Decision Tree, Naive Bayes
4. XGBoost
5. Binary (2 thresholds: Popular >= 46, Popular >= training median)
6. Compare all models
7. Summary
8. Save best model

> **Prereq:** run `00_data_setup.ipynb` first -> `df_popularity_stratified.csv` on Drive.

## 0. Setup

In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
# imports + shared constants (copy from Nb 00)
import os
import pickle
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from collections import Counter
from xgboost import XGBClassifier

from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.feature_selection import SelectKBest, mutual_info_classif
from sklearn.metrics import classification_report, confusion_matrix, ConfusionMatrixDisplay, accuracy_score, f1_score
from sklearn.neighbors import KNeighborsClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.naive_bayes import GaussianNB

import warnings
warnings.filterwarnings('ignore')

SEED            = 42
DRIVE_DATA_PATH = '/content/drive/MyDrive/data-mining-spotify-team3/cleanedData'
MODEL_PATH      = '/content/drive/MyDrive/data-mining-spotify-team3/models'
AUDIO_FEATURES = [
    'danceability', 'energy', 'loudness',
    'speechiness', 'acousticness', 'instrumentalness',
    'liveness', 'valence', 'tempo', 'duration_ms'
]
KEY_FEATURES = ['key', 'mode']
ALL_FEATURES = AUDIO_FEATURES + KEY_FEATURES
LABEL_MAP = {'Low': 0, 'Mid': 1, 'High': 2}

np.random.seed(SEED)

## 1. Load Data

In [ ]:
# load popularity-stratified dataset from Drive (from Nb 00)
df = pd.read_csv(os.path.join(DRIVE_DATA_PATH, 'df_popularity_stratified.csv'))
print(f'Loaded: {df.shape}')
print('\nGlobal popularity class:')
print(df['popularity_class'].value_counts())
print('\nGenre-relative popularity class:')
print(df['genre_popularity_class'].value_counts())
df.head()

## 2. Preprocess & Split

- Scale audio features (StandardScaler).
- Encode labels.
- 60/20/20 train/eval/test split. eval = tuning, test = final report.
- Class balance check (decides SMOTE).
- MI feature selection on train only.

In [ ]:
# scale with StandardScaler. preserve df's index so we can recover raw `popularity` in §5 binary
scaler = StandardScaler()
X = pd.DataFrame(
    scaler.fit_transform(df[ALL_FEATURES]),
    columns=ALL_FEATURES,
    index=df.index
)

# encode labels
y_global = df['popularity_class'].map(LABEL_MAP)
y_genre  = df['genre_popularity_class'].map(LABEL_MAP)

print('Feature matrix shape:', X.shape)
print('Global label distribution:', y_global.value_counts().to_dict())
print('Genre-relative label distribution:', y_genre.value_counts().to_dict())

In [ ]:
# 60/20/20 stratified on global pop label, carry genre-relative target through too
X_trainval, X_test, y_trainval_g, y_test_g, y_trainval_gr, y_test_gr = train_test_split(
    X, y_global, y_genre,
    test_size=0.20, random_state=SEED, stratify=y_global
)
X_train, X_eval, y_train_g, y_eval_g, y_train_gr, y_eval_gr = train_test_split(
    X_trainval, y_trainval_g, y_trainval_gr,
    test_size=0.25, random_state=SEED, stratify=y_trainval_g
)

print(f'Train: {len(X_train):,} ({len(X_train) / len(X):.0%})')
print(f'Eval:  {len(X_eval):,} ({len(X_eval) / len(X):.0%})')
print(f'Test:  {len(X_test):,} ({len(X_test) / len(X):.0%})')

### 2.1 Feature Selection

Rank 12 features by MI vs global popularity. Both targets share same input features, so one selection fits both. Fit on train only (no eval/test leak).

In [ ]:
# MI scorer fit on train only
mi_scorer = SelectKBest(score_func=mutual_info_classif, k='all')
mi_scorer.fit(X_train, y_train_g)

mi_scores = pd.DataFrame({
    'Feature': ALL_FEATURES,
    'MI Score': mi_scorer.scores_
}).sort_values('MI Score', ascending=False).reset_index(drop=True)

print('Mutual information ranking (target = global popularity):')
print(mi_scores.to_string(index=False))

fig, ax = plt.subplots(figsize=(9, 5))
ax.barh(mi_scores['Feature'][::-1], mi_scores['MI Score'][::-1], color='steelblue')
ax.set_xlabel('Mutual Information Score')
ax.set_title('Feature Ranking by MI with Global Popularity (train only)')
plt.tight_layout()
plt.show()

# Tune K here
K = 10
SELECTED_FEATURES = mi_scores['Feature'].head(K).tolist()
print(f'\nSelected top-{K} features: {SELECTED_FEATURES}')
print(f'Dropped: {[f for f in ALL_FEATURES if f not in SELECTED_FEATURES]}')

# project all splits onto selected features
X_train_sel = X_train[SELECTED_FEATURES]
X_eval_sel  = X_eval[SELECTED_FEATURES]
X_test_sel  = X_test[SELECTED_FEATURES]

### 2.2 Class Distribution Check

Nb 00 stratifies by (pop_class x genre) so expect rough balance. If imbalance ratio > 1.5: apply SMOTE on train. Otherwise skip.

In [ ]:
# check imbalance, apply SMOTE if > 1.5x
def check_balance(y, label):
    counts = Counter(y)
    ratio = max(counts.values()) / min(counts.values())
    print(f'{label}: {dict(counts)} | imbalance ratio = {ratio:.2f}')
    return ratio

print('Train set class balance:')
r_g  = check_balance(y_train_g,  'Global popularity        ')
r_gr = check_balance(y_train_gr, 'Genre-relative popularity')

IMBALANCE_THRESHOLD = 1.5

if r_g > IMBALANCE_THRESHOLD or r_gr > IMBALANCE_THRESHOLD:
    from imblearn.over_sampling import SMOTE
    smote = SMOTE(random_state=SEED)
    print(f'\nImbalance detected (threshold={IMBALANCE_THRESHOLD}). Applying SMOTE to training set.')
    if r_g > IMBALANCE_THRESHOLD:
        X_train_g_sel, y_train_g = smote.fit_resample(X_train_sel, y_train_g)
        print(f'  Global target rebalanced: {dict(Counter(y_train_g))}')
    else:
        X_train_g_sel = X_train_sel
    if r_gr > IMBALANCE_THRESHOLD:
        X_train_gr_sel, y_train_gr = smote.fit_resample(X_train_sel, y_train_gr)
        print(f'  Genre-relative target rebalanced: {dict(Counter(y_train_gr))}')
    else:
        X_train_gr_sel = X_train_sel
else:
    print(f'\nBoth targets within {IMBALANCE_THRESHOLD}x threshold - SMOTE not applied.')
    X_train_g_sel  = X_train_sel
    X_train_gr_sel = X_train_sel

## 3. Baseline Classifiers

KNN, Decision Tree, Naive Bayes - tested on both pop targets.

In [ ]:
# fit, 10-fold CV on train, eval check, test preds + classification report + CM
def evaluate(name, model, X_tr, y_tr, X_ev, y_ev, X_te, y_te, target_names=None):
    """
    target_names: class labels for classification report. defaults to 3-class; pass BIN_NAMES for binary.
    """
    if target_names is None:
        target_names = ['Low', 'Mid', 'High']

    model.fit(X_tr, y_tr)

    # 10-fold CV on train (stability estimate)
    cv_scores = cross_val_score(model, X_tr, y_tr, cv=10, scoring='accuracy')
    print(f'\n=== {name} ===')
    print("CV Accuracy (train, 10-fold): %0.2f (+/- %0.2f)" % (cv_scores.mean(), cv_scores.std() * 2))

    # eval-set check
    eval_preds = model.predict(X_ev)
    print(f'Eval Accuracy: {accuracy_score(y_ev, eval_preds):.4f}')

    # test-set final preds (reported in §6)
    test_preds = model.predict(X_te)
    print('\n--- TEST SET ---')
    print(classification_report(y_te, test_preds, target_names=target_names))
    disp = ConfusionMatrixDisplay(
        confusion_matrix(y_te, test_preds),
        display_labels=target_names
    )
    disp.plot(cmap='Blues')
    plt.title(f'{name} - Test Set')
    plt.show()
    return model, test_preds

### 3.1 Target: Global Popularity

In [ ]:
# KNN / DT / NB on global pop target
print('=' * 50)
print('TARGET: GLOBAL POPULARITY')
print('=' * 50)

knn_g, preds_knn_g = evaluate(
    'KNN (k=5) - Global',
    KNeighborsClassifier(n_neighbors=5),
    X_train_g_sel, y_train_g, X_eval_sel, y_eval_g, X_test_sel, y_test_g
)

dt_g, preds_dt_g = evaluate(
    'Decision Tree - Global',
    DecisionTreeClassifier(max_depth=10, random_state=SEED),
    X_train_g_sel, y_train_g, X_eval_sel, y_eval_g, X_test_sel, y_test_g
)

nb_g, preds_nb_g = evaluate(
    'Naive Bayes - Global',
    GaussianNB(),
    X_train_g_sel, y_train_g, X_eval_sel, y_eval_g, X_test_sel, y_test_g
)

### 3.2 Target: Genre-Relative Popularity

In [ ]:
# KNN / DT / NB on genre-relative target (SMOTE-resampled train)
print('=' * 50)
print('TARGET: GENRE-RELATIVE POPULARITY')
print('=' * 50)

knn_gr, preds_knn_gr = evaluate(
    'KNN (k=5) - Genre-Relative',
    KNeighborsClassifier(n_neighbors=5),
    X_train_gr_sel, y_train_gr, X_eval_sel, y_eval_gr, X_test_sel, y_test_gr
)

dt_gr, preds_dt_gr = evaluate(
    'Decision Tree - Genre-Relative',
    DecisionTreeClassifier(max_depth=10, random_state=SEED),
    X_train_gr_sel, y_train_gr, X_eval_sel, y_eval_gr, X_test_sel, y_test_gr
)

nb_gr, preds_nb_gr = evaluate(
    'Naive Bayes - Genre-Relative',
    GaussianNB(),
    X_train_gr_sel, y_train_gr, X_eval_sel, y_eval_gr, X_test_sel, y_test_gr
)

## 4. XGBoost

Gradient-boosted trees. Strongest for tabular weak-signal problems. Handles feature interactions + redundancy (energy/loudness/acousticness multicollinearity from Nb 00) better than baselines.

**Hyperparams** (milder than Nb 01 since popularity signal is weaker):
- `n_estimators=500`, `max_depth=6`, `learning_rate=0.05`
- `subsample=0.9`, `colsample_bytree=0.9` (row + col subsampling)
- `reg_lambda=1.0` (L2)

### 4.1 XGBoost — Global Popularity (3-class)

In [ ]:
# XGBoost on global target (3-class)
xgb_g = XGBClassifier(
    n_estimators=500,
    max_depth=6,
    learning_rate=0.05,
    subsample=0.9,
    colsample_bytree=0.9,
    reg_lambda=1.0,
    objective='multi:softmax',
    num_class=3,
    eval_metric='mlogloss',
    random_state=SEED,
    n_jobs=-1,
)

xgb_g, preds_xgb_g = evaluate(
    'XGBoost - Global',
    xgb_g,
    X_train_g_sel, y_train_g, X_eval_sel, y_eval_g, X_test_sel, y_test_g
)

### 4.2 XGBoost — Genre-Relative Popularity (3-class)

In [ ]:
# XGBoost on genre-relative target (3-class, SMOTE-resampled train)
xgb_gr = XGBClassifier(
    n_estimators=500,
    max_depth=6,
    learning_rate=0.05,
    subsample=0.9,
    colsample_bytree=0.9,
    reg_lambda=1.0,
    objective='multi:softmax',
    num_class=3,
    eval_metric='mlogloss',
    random_state=SEED,
    n_jobs=-1,
)

xgb_gr, preds_xgb_gr = evaluate(
    'XGBoost - Genre-Relative',
    xgb_gr,
    X_train_gr_sel, y_train_gr, X_eval_sel, y_eval_gr, X_test_sel, y_test_gr
)

### 4.3 XGBoost Feature Importance

Which audio features actually drive XGBoost? More trustworthy than §2.1 MI because XGBoost importance accounts for redundancy - if 2 features carry the same signal (e.g. energy + loudness), only one scores high in tree splits.

In [ ]:
# XGBoost FI: global vs genre-relative targets side by side
xgb_fi = pd.DataFrame({
    'Feature': SELECTED_FEATURES,
    'Global Target':        xgb_g.feature_importances_,
    'Genre-Relative Target': xgb_gr.feature_importances_,
}).sort_values('Global Target', ascending=False)

print(xgb_fi.to_string(index=False))

fig, ax = plt.subplots(figsize=(10, 5))
x = np.arange(len(xgb_fi))
width = 0.4
ax.bar(x - width/2, xgb_fi['Global Target'],         width, label='Global Target')
ax.bar(x + width/2, xgb_fi['Genre-Relative Target'], width, label='Genre-Relative Target')
ax.set_xticks(x)
ax.set_xticklabels(xgb_fi['Feature'], rotation=45, ha='right')
ax.set_ylabel('Feature Importance')
ax.set_title('XGBoost Feature Importance: Global vs Genre-Relative Targets')
ax.legend()
plt.tight_layout()
plt.show()

## 5. Binary Classification

3-class is hard with weak signal. Test if collapsing to binary recovers usable accuracy.

**Two thresholds:**

| Target | Threshold | Rationale |
|---|---|---|
| `y_bin_high` | `popularity_class == 'High'` (score >= 46) | matches Nb 00's "noticeably popular". ~1:2 imbalance (3,500 vs 7,000). |
| `y_bin_p50` | raw popularity >= **training-set median** | data-driven. ~50/50 balance by construction. median computed on train only (no leak). |

Both derived from existing splits - no re-splitting.

In [ ]:
# build 2 binary targets from existing split indices
# y_bin_high: Popular = High class (pop >= 46). 1:2 imbalanced.
# derived via split indices (not from y_train_g, which was overwritten by SMOTE for genre-rel target)
y_train_bh = (df.loc[X_train.index, 'popularity_class'] == 'High').astype(int).values
y_eval_bh  = (df.loc[X_eval.index,  'popularity_class'] == 'High').astype(int).values
y_test_bh  = (df.loc[X_test.index,  'popularity_class'] == 'High').astype(int).values

# y_bin_p50: Popular = raw pop >= training median. balanced by construction.
pop_train = df.loc[X_train.index, 'popularity']
pop_eval  = df.loc[X_eval.index,  'popularity']
pop_test  = df.loc[X_test.index,  'popularity']

POP_MEDIAN = pop_train.median()
print(f'Training-set popularity median (threshold for y_bin_p50): {POP_MEDIAN}')

y_train_bp = (pop_train >= POP_MEDIAN).astype(int).values
y_eval_bp  = (pop_eval  >= POP_MEDIAN).astype(int).values
y_test_bp  = (pop_test  >= POP_MEDIAN).astype(int).values

print('\ny_bin_high (Popular = High, score>=46):')
print(f'  train: {dict(Counter(y_train_bh))} | eval: {dict(Counter(y_eval_bh))} | test: {dict(Counter(y_test_bh))}')
print('\ny_bin_p50 (Popular = above training median):')
print(f'  train: {dict(Counter(y_train_bp))} | eval: {dict(Counter(y_eval_bp))} | test: {dict(Counter(y_test_bp))}')

### 5.1 Binary Target: Popular = High (score >= 46)

All 4 model families on the same 3-way split. Uses `X_train_sel` (not SMOTE'd) since binary is independent of 3-class resampling.

In [ ]:
# 4 models on y_bin_high (KNN, DT, NB, XGBoost)
print('=' * 50)
print('BINARY TARGET 1: Popular = High (score >= 46)')
print('=' * 50)

BIN_NAMES = ['Not-Popular', 'Popular']

knn_bh, preds_knn_bh = evaluate(
    'KNN (k=5) - Bin High',
    KNeighborsClassifier(n_neighbors=5),
    X_train_sel, y_train_bh, X_eval_sel, y_eval_bh, X_test_sel, y_test_bh,
    target_names=BIN_NAMES
)

dt_bh, preds_dt_bh = evaluate(
    'Decision Tree - Bin High',
    DecisionTreeClassifier(max_depth=10, random_state=SEED),
    X_train_sel, y_train_bh, X_eval_sel, y_eval_bh, X_test_sel, y_test_bh,
    target_names=BIN_NAMES
)

nb_bh, preds_nb_bh = evaluate(
    'Naive Bayes - Bin High',
    GaussianNB(),
    X_train_sel, y_train_bh, X_eval_sel, y_eval_bh, X_test_sel, y_test_bh,
    target_names=BIN_NAMES
)

xgb_bh = XGBClassifier(
    n_estimators=500, max_depth=6, learning_rate=0.05,
    subsample=0.9, colsample_bytree=0.9, reg_lambda=1.0,
    objective='binary:logistic', eval_metric='logloss',
    random_state=SEED, n_jobs=-1,
)
xgb_bh, preds_xgb_bh = evaluate(
    'XGBoost - Bin High',
    xgb_bh,
    X_train_sel, y_train_bh, X_eval_sel, y_eval_bh, X_test_sel, y_test_bh,
    target_names=BIN_NAMES
)

### 5.2 Binary Target: Popular = above training-set median

Data-driven threshold. Balanced by construction -> cleanest test of whether audio features carry any usable popularity signal.

In [ ]:
# 4 models on y_bin_p50
print('=' * 50)
print(f'BINARY TARGET 2: Popular = above training median ({POP_MEDIAN})')
print('=' * 50)

knn_bp, preds_knn_bp = evaluate(
    'KNN (k=5) - Bin P50',
    KNeighborsClassifier(n_neighbors=5),
    X_train_sel, y_train_bp, X_eval_sel, y_eval_bp, X_test_sel, y_test_bp,
    target_names=BIN_NAMES
)

dt_bp, preds_dt_bp = evaluate(
    'Decision Tree - Bin P50',
    DecisionTreeClassifier(max_depth=10, random_state=SEED),
    X_train_sel, y_train_bp, X_eval_sel, y_eval_bp, X_test_sel, y_test_bp,
    target_names=BIN_NAMES
)

nb_bp, preds_nb_bp = evaluate(
    'Naive Bayes - Bin P50',
    GaussianNB(),
    X_train_sel, y_train_bp, X_eval_sel, y_eval_bp, X_test_sel, y_test_bp,
    target_names=BIN_NAMES
)

xgb_bp = XGBClassifier(
    n_estimators=500, max_depth=6, learning_rate=0.05,
    subsample=0.9, colsample_bytree=0.9, reg_lambda=1.0,
    objective='binary:logistic', eval_metric='logloss',
    random_state=SEED, n_jobs=-1,
)
xgb_bp, preds_xgb_bp = evaluate(
    'XGBoost - Bin P50',
    xgb_bp,
    X_train_sel, y_train_bp, X_eval_sel, y_eval_bp, X_test_sel, y_test_bp,
    target_names=BIN_NAMES
)

## 6. Compare All Models (Test Set)

Final numbers. 4 model families (KNN, DT, NB, XGBoost) x 4 targets (global 3-class, genre-rel 3-class, binary >=High, binary >=median).

In [ ]:
# build 4 per-target test-score tables (acc / macro F1 / weighted F1)
def scores(preds, y_te):
    return {
        'Accuracy':    round(accuracy_score(y_te, preds), 4),
        'Macro F1':    round(f1_score(y_te, preds, average='macro'),    4),
        'Weighted F1': round(f1_score(y_te, preds, average='weighted'), 4),
    }

# 3-class global
df_global = pd.DataFrame([
    {'Model': 'KNN',           **scores(preds_knn_g, y_test_g)},
    {'Model': 'Decision Tree', **scores(preds_dt_g,  y_test_g)},
    {'Model': 'Naive Bayes',   **scores(preds_nb_g,  y_test_g)},
    {'Model': 'XGBoost',       **scores(preds_xgb_g, y_test_g)},
])

# 3-class genre-relative
df_genre_rel = pd.DataFrame([
    {'Model': 'KNN',           **scores(preds_knn_gr, y_test_gr)},
    {'Model': 'Decision Tree', **scores(preds_dt_gr,  y_test_gr)},
    {'Model': 'Naive Bayes',   **scores(preds_nb_gr,  y_test_gr)},
    {'Model': 'XGBoost',       **scores(preds_xgb_gr, y_test_gr)},
])

# binary >= High (score >= 46)
df_bin_high = pd.DataFrame([
    {'Model': 'KNN',           **scores(preds_knn_bh, y_test_bh)},
    {'Model': 'Decision Tree', **scores(preds_dt_bh,  y_test_bh)},
    {'Model': 'Naive Bayes',   **scores(preds_nb_bh,  y_test_bh)},
    {'Model': 'XGBoost',       **scores(preds_xgb_bh, y_test_bh)},
])

# binary >= training median
df_bin_p50 = pd.DataFrame([
    {'Model': 'KNN',           **scores(preds_knn_bp, y_test_bp)},
    {'Model': 'Decision Tree', **scores(preds_dt_bp,  y_test_bp)},
    {'Model': 'Naive Bayes',   **scores(preds_nb_bp,  y_test_bp)},
    {'Model': 'XGBoost',       **scores(preds_xgb_bp, y_test_bp)},
])

print('Target: Global Popularity (3-class)')
print(df_global.to_string(index=False))
print('\nTarget: Genre-Relative Popularity (3-class)')
print(df_genre_rel.to_string(index=False))
print('\nTarget: Binary - Popular = High (score >= 46)')
print(df_bin_high.to_string(index=False))
print(f'\nTarget: Binary - Popular >= training median ({POP_MEDIAN})')
print(df_bin_p50.to_string(index=False))

In [ ]:
# 2x2 grid: one panel per target, 4 models x 3 metrics
fig, axes = plt.subplots(2, 2, figsize=(14, 9), sharey=True)

panels = [
    (axes[0, 0], df_global,    '3-class - Global Popularity'),
    (axes[0, 1], df_genre_rel, '3-class - Genre-Relative Popularity'),
    (axes[1, 0], df_bin_high,  'Binary - Popular = High (>=46)'),
    (axes[1, 1], df_bin_p50,   f'Binary - Popular >= median ({POP_MEDIAN})'),
]

for ax, df_res, title in panels:
    x = np.arange(len(df_res))
    width = 0.25
    ax.bar(x - width, df_res['Accuracy']    * 100, width, label='Accuracy')
    ax.bar(x,         df_res['Macro F1']    * 100, width, label='Macro F1')
    ax.bar(x + width, df_res['Weighted F1'] * 100, width, label='Weighted F1')
    ax.set_title(title)
    ax.set_xticks(x)
    ax.set_xticklabels(df_res['Model'], rotation=15, ha='right')
    ax.set_ylabel('Score (%)')
    ax.set_ylim(0, 100)
    ax.axhline(50, color='gray', linestyle='--', linewidth=0.8, alpha=0.6)
    ax.legend(fontsize=8)

plt.suptitle('All Models Across All Targets', fontsize=14)
plt.tight_layout()
plt.show()

## 7. Summary

**Pipeline**
1. 60/20/20 stratified split.
2. MI feature selection on train only (K=10, drops tempo + duration_ms -> MI=0 with popularity).
3. Class-balance check + conditional SMOTE (1.5x threshold). genre-rel triggered it, global didn't.
4. 4 model families x 4 targets. no neural net - audio signal too weak (tested MLP, no learning).
5. 10-fold CV on train, eval for tuning, test for final numbers.

**Findings:**

| Target | Best | Acc | Baseline | Real Lift |
|---|---|---|---|---|
| Global 3-class | XGBoost | 39.2% | 33.3% | +5.9pp ok |
| Genre-rel 3-class | XGBoost | 45.9% (macro F1 0.38) | 33.3% | majority-class bias inflates |
| Binary >=46 | XGBoost | 63.7% | **66.7% majority** | **-3pp (accuracy paradox)** |
| **Binary >=median** | **XGBoost** | **55.2%** | **50%** | **+5.2pp (cleanest signal)** |

**Takeaway:** audio = weak but real predictor of popularity. Fine-grained (Low/Mid/High) at the edge of what 10 features can do. Binary recovers cleaner separation. Binary >=46 LOOKS strong but LOSES to majority baseline - model just says "not popular" 80% of the time.

**Baselines:**
- 3-class balanced: 33.3%
- Binary >=46: ~67% majority (must exceed to mean anything)
- Binary >=median: 50%

**Pickle handoff:** best binary model (XGBoost on y_bin_p50) + StandardScaler + feature list + POP_MEDIAN threshold saved to Drive for Nb 03's optional popularity-aware re-ranker.

---
## 8. Save Trained Model

Export binary >= median XGBoost + scaler + feature list + threshold for Nb 03 to load.

In [ ]:
# save XGBoost binary >= median + scaler + feature list + threshold for Nb 03
os.makedirs(MODEL_PATH, exist_ok=True)

# model alone isn't enough - xgb_bp trained on StandardScaler-transformed features,
# so Nb 03 needs scaler + feature list to reproduce identical inputs.
# POP_MEDIAN lets downstream code decode the binary target semantically.
artifacts = {
    'pop_xgb_binary_p50.pkl':   xgb_bp,
    'pop_scaler.pkl':           scaler,
    'pop_feature_list.pkl':     SELECTED_FEATURES,
    'pop_median_threshold.pkl': POP_MEDIAN,
}

for fname, obj in artifacts.items():
    with open(os.path.join(MODEL_PATH, fname), 'wb') as f:
        pickle.dump(obj, f)
    print(f'  saved: {os.path.join(MODEL_PATH, fname)}')